In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 4: Umbrella Sampling

Consider a particle moving in a 1D double-well potential with the following form:

\begin{equation}
 U(x) = a(x^2-b^2)^2 + cx
\end{equation}

In [ ]:
# Define the double-well potential

# Parameters
a = 4.0   # barrier height
b = 1.0   # position of symmetric minima
c = 1.0   # tilt parameter to make one minimum lower

def potential(x, a=a, b=b, c=c):
    U = a*(x**2 - b**2)**2 + c*x
    U_shifted = U - np.min(U)
    return U_shifted

x_vals = np.linspace(-1.6, 1.5, 500)
PES = potential(x_vals)

plt.figure(figsize=(8,5))
plt.plot(x_vals, PES, 'k-', lw=2)
plt.xlabel('x')
plt.ylabel('Energy')
plt.title('Asymmetric Double-Well')
plt.show()

The two minimas of the potential energy surface are separated by a barrier. 

1. Why is the free energy profile identical (up to a constant) to the potential energy profile in this 1D system?

2. Why is it difficult to compute free energy differences between the two minima using standard molecular dynamics?

To achieve more accurate estimates of free energy differences $\Delta F$ we can the **umbrella sampling** scheme suggested by Torrie and Valleau.

In umbrella sampling simulations, normally, an additional harmonic biasing potential is applied on the chosen raction coordiante $\xi$:

$U_{\rm US}(x) = \frac{1}{2} k (\xi(x)-\xi_0)^2$

where $k$ is the force constant and $\xi_0$ the center of the harmonic restraint.

In our case, the reaction coordinate is simply the position of the particle, $\xi = x$.


Normally, to reconstruct the free energy surface along $\xi$ with umbrella sampling, multiple simultions ("windows") are performed, each with the harmonic potential centered at a different value of $\xi$, allowing the system to overcome free energy barriers.<br><br>

Thus, setting up umbrella sampling requires defyining:
- the centers of the umbrella windows $\xi_0$
- the force constant $k$

### Defition of umbrella windows centers

3. The umbrella windows must be placed such that adjacent windows have sufficient overlap. Why is this the case?

For complex systems, the free energy surface is not known a priori, so the appropriate number and placement of the umbrella windows must be determined form preliminary explorations of the reaction coordiante.
 
In this simple case, the potential is known, and therefore the physically relevant region of the reaction coordinate is also known.
Thus, we can use equispaced windows with a distant $\Delta x = 0.5$, from $x=-1.5$ to $x=1.5$.

In [ ]:
# umbrella sampling windows
delta_x = 0.5
centers = np.arange(-1.5, 1.5+delta_x, delta_x)  

### Definition of the force constant

In principle, the method will give the correct answer independently of the umbrella potential that is used.
However, its efficiency strongly depends on the choice of the umbrella potentials. </div>

4. What do you expect to happen if you increase or reduce the value of $k$?

### Umbrella Sampling simulations </h2>

In standard MD: $F(x) = -dU/dx$, where $U$ is the potential of the system. <br><br>

In umbrella sampling:

$U_{\rm tot}(x) = U(x) + U_{\rm US}(x)$


5. What is the total force in an umbrella sampling simulation?
Starting from the algorithm of the Velocity Verlet integrator with Andersen thermostat of Exercise 1, update it to run umbrella sampling simulations (modify the function of the force from umbrella sampling accordingly). 

In [ ]:
# Define forces for Velocity Verlet integration #

# -------- Double-well force --------
def force_dw(x, a=a, b=b, c=c):
    """Force from the asymmetric double-well potential F = -dU/dx."""
    return -(4*a*x*(x**2 - b**2) + c)


# -------- Umbrella force --------
def force_umbrella(x, k_us, x0_us):
    # start implementation
    F_us = 
    return F_us
    # end implementation

# -------- Total force --------
def total_force(x, k_us, x0_us):
    return force_dw(x) + force_umbrella(x, k_us, x0_us)

In [ ]:
# --- Velocity Verlet integrator with Andersen thermostat ---

def velocity_verlet_umbrella(x0, v0, m, dt, n_steps, T=1., nu=0.1, k_us=0, x0_us=0):
    """
    Integrate the motion of a particle using Velocity Verlet with Andersen thermostat (NVT ensemble).
    
    Parameters:
        x0, v0 : initial position and velocity
        m      : mass
        dt     : time step
        n_steps: number of integration steps
        T      : temperature (for thermostat) in reduced units
        nu     : collision frequency for Andersen thermostat
        k_us   : umbrella sampling force constant
        x0_us  : umbrella sampling center position
    Returns:
        x_traj, v_traj, t_traj : arrays of positions, velocities, and times
    """
    kB = 1.0  # Boltzmann constant in reduced units

    x = x0
    v = v0
    F = total_force(x, k_us, x0_us)
    
    x_traj = np.zeros(n_steps)
    v_traj = np.zeros(n_steps)
    t_traj = np.zeros(n_steps)
    
    for i in range(n_steps):
        # --- Velocity Verlet steps ---
        # start implementation
        # --- Andersen thermostat ---
        if np.random.rand() < nu * dt:
            # Reassign velocity from Maxwell-Boltzmann distribution
            v = None
        # end implementation

        # Store trajectories
        x_traj[i] = x
        v_traj[i] = v
        t_traj[i] = i*dt
        
    return t_traj, x_traj, v_traj

Now, we can run the umbrella sampling simulations in the NVT ensemble, each with the bias harmonic potential centered in $x_0$ from -1.5 to 1.5.

In [ ]:
# number of steps of the simulation
n_steps = 5000000

# store trajectory for each umbrella center
trajectories = np.zeros((len(centers), n_steps))

# umbrella force constant to be tuned for good sampling
k_us = 100

# Run US simulations for each umbrella center
for i,x0_us in enumerate(centers):
    print(f"Running umbrella sampling simulation for center at x={x0_us:.2f}")
    t, x, v = velocity_verlet_umbrella(
    x0=x0_us,         # start the simulation in the center of the umbrella
    v0=0.0,
    m=1.0,
    dt=0.001,
    n_steps=n_steps,
    T=1,             
    nu=0.1,
    k_us=k_us,         # umbrella strength
    x0_us=x0_us       # umbrella center
    )

    trajectories[i] = x  # Store the trajectory for analysis


In [ ]:
# Plot histogram for each umbrella window
for i, x0_us in enumerate(centers):
    plt.hist(trajectories[i],bins=20,density=True,alpha=0.5,label=f"Center = {x0_us:.2f}")

plt.xlabel("x coordinate")
plt.ylabel("Biased probability density")
plt.title("Biased Histograms from Umbrella Sampling Windows")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

6. Do the probability distributions from adjacent umbrella windows show sufficient overlap? How does the overlap change when increasing or decreasing the force constant $k$? Which range of $k$ values gives you a good overlap between adjacent distributions?

### Reconstruction of the free energy surface

The free energy surface along a reaction coordinate $x$ can be obtained from the **Boltzmann distributions** of $x$ in the NVT ensemble:

${\rm F}(x) = - k_{\rm B}*T * \ln \rho(x)$

where $\rho(x)$ is the probability density of observing the system at position $x$.<br><br>

However, the histograms we collected from umbrella sampling correspond to **biased, non-Boltzmann distributions**.

This is because we sampled $x$ according to a biased potential $U_{\rm tot}(x) = U(x) + U_{\rm US}(x)$.<br><br>

To recover the unbias Boltzmann distribution and determine the 1D free energy surface, we need to correct for this bias. This is exactly what post-processing methods like WHAM (Weighted Histogram Analysis Method) do.

In [ ]:
# Download WHAM release 2.1.0 from the Grossfield lab http://membrane.urmc.rochester.edu/?page_id=79
!wget http://membrane.urmc.rochester.edu/sites/default/files/wham/wham-release-2.1.0.tgz

# Extract the tarball
!tar -xvf wham-release-2.1.0.tgz

# install WHAM
!mkdir wham/build
!cd  wham/build && cmake .. &&cmake --build .

In [ ]:
# You might need to change permissions to make the wham executable runnable
#!chmod +x wham/wham

To use WHAM, we need to prepare a file for each umbrella sampling window contaning two colums: time and $x$ values sampled during umbrella sampling.

In [ ]:
# Create a folder to store the umbrella data
os.makedirs("umbrella_sampling", exist_ok=True)

# Time step and number of steps
dt=0.001
n_steps = trajectories.shape[1]

# Loop over windows and save files
for i, x0_us in enumerate(centers):
    t = np.arange(n_steps) * dt
    x = trajectories[i]
    data = np.column_stack((t, x))
    
    filename = f"umbrella_sampling/window_{x0_us}.dat"
    np.savetxt(filename, data, fmt="%.6f", comments='', delimiter=' ')

Then, we need to define a metadata file, which lists for each umbrella window:
- the corresponding file name
- the umbrella center $x_0$ 
- the force constant $k$.

In [ ]:
# WHAM PMF is calculated from different umbrella windows
file_names = [f"umbrella_sampling/window_{x0_us}.dat" for x0_us in centers]

# Prepare WHAM input file
wham_file = pd.DataFrame({
    'file names': file_names,
    'x_0': centers,
    'k': [k_us]*len(centers)
})

# Save the metadata file for WHAM
wham_file.to_csv(f'umbrella_sampling/wham_us', index=False, header=False, sep=' ')

Now, we can run WHAM by specifying:
- the units (here, reduced units with $k_B=1$)
- the minimum and maximum of the histogram
- the number of bins
- the tolerance for convergence
- the temperature (in reduced units, T=1)
- the perodicity of the coordinate (0 because the coordinate is not periodic)
- the name of the metadata file
- the name of the output file

In [ ]:
!  wham/build/wham units lj -1.6 1.5 80 0.0001 1 0 umbrella_sampling/wham_us umbrella_sampling/fes_us

Finally, we can compare the free energy surface obtained from umbrella sampling simulations and WHAM with the origial potential.


In [ ]:
fes_us = pd.read_csv(f'umbrella_sampling/fes_us', comment='#', header=None, sep='\t')

plt.plot(fes_us[0], fes_us[1], label='US FES', color='blue', lw=2)
plt.plot(x_vals, PES, 'k-', lw=2, label='True PES')
plt.xlabel('x')
plt.ylabel('Energy')
plt.legend()
plt.grid()
plt.show()

7. Does the free energy profile match the potential energy surface? How does the quality of overlap between adjacent umbrella windows affect the free energy profile?

(Small differences might be due to an incomplete sampling, you can try increasing the number of steps of the umbrella sampling simulations)